# Pig behaviour baseline, v2 (continues the first notebook)

Run this after the original notebook, in the same Kaggle session, so `/kaggle/working/reconstructed_annotations.csv` exists.

What changes compared with v1:
1. The 43,232 interpolated rows are no longer thrown away wholesale. They are kept only when the label and box are likely to be right (gap, IoU and label-agreement gate). Each row gets a tier: `stored`, `gated`, or `rejected`.
2. Evaluation is leave-one-day-out over all 7 recording days. There is no single validation day.
3. Cheap baselines come first: majority class, box geometry and motion features, then frozen-embedding logistic regression. Fine-tuning has to beat these before it is worth the GPU time.
4. A coarse ethogram (from your `behavior_mapping_draft.json`) is scored next to the 7 classes. It is a draft and needs domain review.

In [ ]:
import os, json, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
warnings.filterwarnings("ignore")
try:
    display
except NameError:
    display = print

WORK = Path(os.environ.get("AI4E_WORK", "/kaggle/working"))
DATA_ROOT = Path(os.environ.get(
    "AI4E_DATA", "/kaggle/input/datasets/jackbyte/edinburgh-pig-behaviour-annotated"))
SEED = 42
W, H = 1280, 720

CLASSES7 = ["sleep", "investigating", "eat", "standing", "lying", "walk", "drink"]
# Draft grouping from behavior_mapping_draft.json (needs domain review).
COARSE = {"sleep": "resting", "lying": "resting", "standing": "stationary",
          "walk": "locomotion", "eat": "feeding", "drink": "drinking",
          "investigating": "exploring"}

# Gate for rows that were not manually stored. Tune with the audit sheet below.
MAX_GAP = 20      # annotation steps (about 10 s) between bracketing stored records
MIN_IOU = 0.5     # box overlap between the two bracketing stored records
MAX_TRAIL = 10    # steps a last stored record may be held forward

## 1. Gate the reconstructed rows

A reconstructed row sits between two stored (human) records. It is probably fine when the two records are close in time, the pig barely moved between them, and the label is the same on both sides. It is risky otherwise. Rows held forward past the last record have no right-hand check, so they only get a length cap.

In [ ]:
labels = pd.read_csv(WORK / "reconstructed_annotations.csv", dtype={"pig_id": str})
labels["recording_date"] = labels["clip_id"].str.split("/").str[0]
labels = labels.sort_values(["clip_id", "pig_id", "annotation_index"]).reset_index(drop=True)

def box_iou(a, b):
    """a, b: (n,4) arrays of x, y, w, h. Row-wise IoU."""
    ax2, ay2 = a[:, 0] + a[:, 2], a[:, 1] + a[:, 3]
    bx2, by2 = b[:, 0] + b[:, 2], b[:, 1] + b[:, 3]
    iw = np.clip(np.minimum(ax2, bx2) - np.maximum(a[:, 0], b[:, 0]), 0, None)
    ih = np.clip(np.minimum(ay2, by2) - np.maximum(a[:, 1], b[:, 1]), 0, None)
    inter = iw * ih
    union = a[:, 2] * a[:, 3] + b[:, 2] * b[:, 3] - inter
    return np.where(union > 0, inter / union, 0.0)

def add_gate_columns(df):
    gap = np.full(len(df), np.nan); ext = np.full(len(df), np.nan)
    iou_lr = np.full(len(df), np.nan); agree = np.zeros(len(df), bool)
    has_right = np.zeros(len(df), bool)
    for _, ix in df.groupby(["clip_id", "pig_id"], sort=False).indices.items():
        g = df.iloc[ix]
        idx = g["annotation_index"].to_numpy()
        st = (g["annotation_source"] == "stored").to_numpy()
        sidx = idx[st]
        sbox = g.loc[st, ["x", "y", "width", "height"]].to_numpy(float)
        sbeh = g.loc[st, "behavior"].to_numpy()
        L = np.searchsorted(sidx, idx, side="right") - 1
        R = np.searchsorted(sidx, idx, side="left")
        hr = R < len(sidx)
        L = np.clip(L, 0, len(sidx) - 1); Rc = np.clip(R, 0, len(sidx) - 1)
        gap[ix] = np.where(hr, sidx[Rc] - sidx[L], np.nan)
        ext[ix] = np.where(hr, np.nan, idx - sidx[L])
        iou_lr[ix] = np.where(hr, box_iou(sbox[L], sbox[Rc]), np.nan)
        agree[ix] = sbeh[L] == sbeh[Rc]
        has_right[ix] = hr
    out = df.copy()
    out["gap"], out["trail_ext"], out["iou_lr"] = gap, ext, iou_lr
    out["label_agree"], out["has_right"] = agree, has_right
    return out

def assign_tier(df, max_gap=MAX_GAP, min_iou=MIN_IOU, max_trail=MAX_TRAIL):
    stored = df["annotation_source"].eq("stored")
    inter_ok = (df["has_right"] & (df["gap"] <= max_gap)
                & (df["iou_lr"] >= min_iou) & df["label_agree"])
    trail_ok = (~df["has_right"]) & (df["trail_ext"] <= max_trail)
    return np.where(stored, "stored", np.where(inter_ok | trail_ok, "gated", "rejected"))

labels = add_gate_columns(labels)
labels["tier"] = assign_tier(labels)

print("Rows by tier:")
display(labels["tier"].value_counts().rename("rows"))
print("\nStored+gated rows per baseline class (stored only in brackets):")
b7 = labels[labels["behavior"].isin(CLASSES7)]
display(pd.DataFrame({
    "kept": b7[b7.tier != "rejected"].groupby("behavior").size(),
    "stored_only": b7[b7.tier == "stored"].groupby("behavior").size(),
    "v1_accepted_incl_trailing": b7[b7.annotation_source != "interpolated_box_forward_label"]
        .groupby("behavior").size(),
}).reindex(CLASSES7))

In [ ]:
# Sensitivity of the gate: how much data survives, and how many rows per class per day.
rows = []
for mg in (5, 10, 20, 40):
    for mi in (0.3, 0.5, 0.7):
        t = assign_tier(labels, mg, mi, MAX_TRAIL)
        k = labels[(t != "rejected") & labels["behavior"].isin(CLASSES7)]
        per_day = k.groupby(["behavior", "recording_date"]).size().unstack(fill_value=0)
        rows.append({"max_gap": mg, "min_iou": mi, "kept_rows": len(k),
                     "min_class_total": int(per_day.sum(1).reindex(CLASSES7, fill_value=0).min()),
                     "day_class_cells_empty": int((per_day.reindex(CLASSES7, fill_value=0) == 0).sum().sum())})
display(pd.DataFrame(rows))

### Audit the gate by eye

Before trusting the gate, look at a random sample of `gated` rows and count how many boxes really sit on the labelled pig. Report that pass rate in the paper. If it is low, tighten `MAX_GAP` and `MIN_IOU`.

In [ ]:
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def audit_sheet(df, n=24, seed=SEED, out=WORK / "gated_audit_sheet.png"):
    sample = df[df["tier"] == "gated"].sample(n, random_state=seed)
    cols = 6; rows_ = math.ceil(n / cols)
    fig, axes = plt.subplots(rows_, cols, figsize=(cols * 3.2, rows_ * 2.4))
    for ax, (_, r) in zip(axes.ravel(), sample.iterrows()):
        cap = cv2.VideoCapture(str(DATA_ROOT / r["clip_id"] / "color.mp4"))
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(r["raw_frame_index"]))
        ok, fr = cap.read(); cap.release()
        if ok:
            ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
            ax.add_patch(Rectangle((r["x"], r["y"]), r["width"], r["height"],
                                   fill=False, edgecolor="red", linewidth=1.5))
        ax.set_title(f"{r['clip_id'][-17:]} p{r['pig_id']} i{int(r['annotation_index'])}\n{r['behavior']}", fontsize=6)
        ax.axis("off")
    plt.tight_layout(); plt.savefig(out, dpi=110); plt.show()
    print("Saved", out, "| count correct boxes and record the pass rate.")

if (DATA_ROOT).exists():
    audit_sheet(labels)

## 2. Box geometry and motion features

No pixels here. These are posture and activity cues that a single crop cannot give: box shape (lying vs standing), speed from the real capture timestamps (not the 30 fps video clock), position in the pen (feeder and drinker zones), and crowding (huddles).

Speed is computed only from reliable boxes (`stored` or `gated`), so a long interpolated stretch does not fake a smooth velocity.

In [ ]:
def pair_iou_matrix(b):
    x2, y2 = b[:, 0] + b[:, 2], b[:, 1] + b[:, 3]
    iw = np.clip(np.minimum(x2[:, None], x2[None]) - np.maximum(b[:, 0][:, None], b[:, 0][None]), 0, None)
    ih = np.clip(np.minimum(y2[:, None], y2[None]) - np.maximum(b[:, 1][:, None], b[:, 1][None]), 0, None)
    inter = iw * ih
    area = b[:, 2] * b[:, 3]
    return inter / np.maximum(area[:, None] + area[None] - inter, 1e-9)

def build_features(df, half_window=2):
    df = df.reset_index(drop=True)
    f = pd.DataFrame(index=df.index)
    f["cx"] = (df["x"] + df["width"] / 2) / W
    f["cy"] = (df["y"] + df["height"] / 2) / H
    f["w"], f["h"] = df["width"] / W, df["height"] / H
    f["log_area"] = np.log(df["width"] * df["height"])
    f["aspect"] = df["width"] / df["height"]

    speed = np.full(len(df), np.nan)
    reliable = (df["tier"] != "rejected").to_numpy()
    for _, ix in df.groupby(["clip_id", "pig_id"], sort=False).indices.items():
        g = df.iloc[ix]
        cx, cy = (g["x"] + g["width"] / 2).to_numpy(), (g["y"] + g["height"] / 2).to_numpy()
        t = g["elapsed_seconds"].to_numpy(); ok = reliable[ix]; n = len(g)
        lo = np.clip(np.arange(n) - half_window, 0, n - 1)
        hi = np.clip(np.arange(n) + half_window, 0, n - 1)
        dt = t[hi] - t[lo]
        d = np.hypot(cx[hi] - cx[lo], cy[hi] - cy[lo])
        v = np.where(ok[lo] & ok[hi] & (dt > 0), d / np.where(dt > 0, dt, 1), np.nan)
        speed[ix] = v
    f["speed_px_s"] = speed

    nn = np.full(len(df), np.nan); mx = np.full(len(df), np.nan)
    for _, ix in df.groupby(["clip_id", "annotation_index"]).indices.items():
        b = df.iloc[ix][["x", "y", "width", "height"]].to_numpy(float)
        c = b[:, :2] + b[:, 2:] / 2
        d = np.hypot(c[:, None, 0] - c[None, :, 0], c[:, None, 1] - c[None, :, 1])
        np.fill_diagonal(d, np.inf)
        iou = pair_iou_matrix(b); np.fill_diagonal(iou, 0)
        nn[ix] = d.min(1); mx[ix] = iou.max(1)
    f["nearest_pig_px"], f["max_overlap_other"] = nn, mx
    return f

kept = labels[(labels["tier"] != "rejected") & labels["behavior"].isin(CLASSES7)].reset_index(drop=True)
full_feats = build_features(labels)                         # features need the whole series
feat_cols = list(full_feats.columns)
labels_feat = pd.concat([labels.reset_index(drop=True), full_feats], axis=1)
kept = labels_feat[(labels_feat["tier"] != "rejected") & labels_feat["behavior"].isin(CLASSES7)].reset_index(drop=True)
kept["coarse"] = kept["behavior"].map(COARSE)
print("Kept rows for modelling:", len(kept))
display(kept[feat_cols].describe().T[["mean", "std", "min", "max"]].round(3))

## 3. Leave-one-day-out evaluation

Each of the 7 recording days is held out once. The primary score is on `stored` rows only (human-placed records). A second score on all kept rows shows what the gate adds. Macro-F1 uses only the classes present in the held-out day, so a missing class does not silently count as zero.

In [ ]:
class Weighted:
    """Balanced sample weights for models without class_weight."""
    def __init__(self, model): self.model = model
    def fit(self, X, y):
        self.model.fit(X, y, sample_weight=compute_sample_weight("balanced", y)); return self
    def predict(self, X): return self.model.predict(X)

def lodo(X, y, groups, tiers, make_model, name):
    groups, tiers, y = np.asarray(groups), np.asarray(tiers), np.asarray(y)
    folds, P = [], np.empty(len(y), dtype=object)
    for day in sorted(set(groups)):
        te = groups == day
        model = make_model(); model.fit(X[~te], y[~te])
        P[te] = model.predict(X[te])
        for scope, m in (("stored", te & (tiers == "stored")), ("all_kept", te)):
            if m.sum():
                labs = np.unique(y[m])
                folds.append({"model": name, "day": day, "scope": scope, "n": int(m.sum()),
                              "macro_f1": f1_score(y[m], P[m], labels=labs, average="macro", zero_division=0),
                              "accuracy": accuracy_score(y[m], P[m])})
    folds = pd.DataFrame(folds)
    return folds, P

def summarize(folds_list):
    allf = pd.concat(folds_list)
    return (allf.groupby(["model", "scope"])
                .agg(mean_fold_macro_f1=("macro_f1", "mean"), std_fold=("macro_f1", "std"),
                     mean_fold_accuracy=("accuracy", "mean"))
                .round(3).reset_index())

def recall_table(y, P, tiers, classes):
    m = np.asarray(tiers) == "stored"
    cm = confusion_matrix(np.asarray(y)[m], P[m], labels=classes)
    return pd.DataFrame(cm, index=classes, columns=classes), pd.Series(
        np.diag(cm) / np.maximum(cm.sum(1), 1), index=classes, name="recall")

y7, y_coarse = kept["behavior"].to_numpy(), kept["coarse"].to_numpy()
groups, tiers = kept["recording_date"].to_numpy(), kept["tier"].to_numpy()
Xg = kept[feat_cols].to_numpy(float)

all_folds, preds = [], {}
for task, y in (("7class", y7), ("coarse", y_coarse)):
    for nm, mk in (
        ("majority", lambda: DummyClassifier(strategy="most_frequent")),
        ("stratified_random", lambda: DummyClassifier(strategy="stratified", random_state=SEED)),
        ("geometry_motion_HGB", lambda: Weighted(HistGradientBoostingClassifier(
            max_depth=4, learning_rate=0.08, max_iter=150, random_state=SEED))),
    ):
        f, P = lodo(Xg, y, groups, tiers, mk, f"{task}:{nm}")
        all_folds.append(f); preds[f"{task}:{nm}"] = P

display(summarize(all_folds))

In [ ]:
# Where does the geometry model fail? (stored rows, pooled over days)
cm, rec = recall_table(y7, preds["7class:geometry_motion_HGB"], tiers, CLASSES7)
display(cm); display(rec.round(2))

## 4. Frozen-embedding baseline

Pretrained MobileNetV3-Small features (the same backbone as v1) with no fine-tuning, then logistic regression. If this matches the fine-tuned model, the fine-tuning in v1 adds nothing but overfitting. Embeddings are computed once per clip and cached. Turn on the GPU in Kaggle first.

Crops use the same recipe as v1: box plus 10% margin, padded to square, 224 px.

In [ ]:
EMB_DIR = WORK / "emb_v2"; EMB_DIR.mkdir(exist_ok=True)
MARGIN = 0.10
PAD_RGB = (124, 116, 104)

def crop_square(frame_bgr, x, y, w, h, size=224):
    H_, W_ = frame_bgr.shape[:2]
    x0 = int(max(0, math.floor(x - MARGIN * w))); y0 = int(max(0, math.floor(y - MARGIN * h)))
    x1 = int(min(W_, math.ceil(x + w * (1 + MARGIN)))); y1 = int(min(H_, math.ceil(y + h * (1 + MARGIN))))
    if x1 <= x0 or y1 <= y0:
        return None
    c = cv2.cvtColor(frame_bgr[y0:y1, x0:x1], cv2.COLOR_BGR2RGB)
    s = max(c.shape[:2]); canvas = np.empty((s, s, 3), np.uint8); canvas[:] = PAD_RGB
    t, l = (s - c.shape[0]) // 2, (s - c.shape[1]) // 2
    canvas[t:t + c.shape[0], l:l + c.shape[1]] = c
    return cv2.resize(canvas, (size, size), interpolation=cv2.INTER_AREA)

def make_torch_embedder():
    import torch
    from torchvision import models
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    net.classifier = torch.nn.Identity(); net.eval().to(dev)
    mean = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
    @torch.no_grad()
    def embed(batch_uint8):                       # (n, 224, 224, 3) RGB uint8
        x = torch.from_numpy(batch_uint8).to(dev).permute(0, 3, 1, 2).float() / 255
        return net((x - mean) / std).cpu().numpy().astype(np.float32)
    print("Embedding device:", dev)
    return embed

def embed_rows(df, embed_fn, batch=128):
    """Returns (n, d) embeddings aligned with df rows. Reads each video once, in order."""
    out = {}
    for clip_id, g in df.groupby("clip_id"):
        cache = EMB_DIR / (clip_id.replace("/", "_") + ".npz")
        if cache.exists():
            z = np.load(cache, allow_pickle=True)
            have = dict(zip(map(tuple, z["keys"]), z["emb"]))
        else:
            have = {}
        need = g[[(r.pig_id, int(r.annotation_index)) not in have for r in g.itertuples()]]
        if len(need):
            by_frame = {}
            for r in need.itertuples():
                by_frame.setdefault(int(r.raw_frame_index), []).append(r)
            cap = cv2.VideoCapture(str(DATA_ROOT / clip_id / "color.mp4"))
            keys, crops, fi = [], [], 0
            last = max(by_frame)
            while fi <= last:
                if fi in by_frame:
                    ok, fr = cap.read()
                    if not ok: break
                    for r in by_frame[fi]:
                        c = crop_square(fr, r.x, r.y, r.width, r.height)
                        if c is not None:
                            keys.append((r.pig_id, int(r.annotation_index))); crops.append(c)
                else:
                    if not cap.grab(): break
                fi += 1
            cap.release()
            embs = [embed_fn(np.stack(crops[i:i + batch])) for i in range(0, len(crops), batch)]
            for k, e in zip(keys, np.concatenate(embs) if embs else []):
                have[k] = e
            kk = list(have.keys())
            np.savez(cache, keys=np.array(kk, dtype=object), emb=np.stack([have[k] for k in kk]))
        d = next(iter(have.values())).shape[0]
        for r in g.itertuples():
            out[r.Index] = have.get((r.pig_id, int(r.annotation_index)), np.full(d, np.nan, np.float32))
    return np.stack([out[i] for i in df.index])

RUN_EMBEDDINGS = DATA_ROOT.exists()
if RUN_EMBEDDINGS:
    emb = embed_rows(kept, make_torch_embedder())
    ok_rows = ~np.isnan(emb).any(1)
    print("Embedded", int(ok_rows.sum()), "of", len(kept), "rows")

In [ ]:
if RUN_EMBEDDINGS:
    k2 = kept[ok_rows].reset_index(drop=True); E = emb[ok_rows]
    g2, t2 = k2["recording_date"].to_numpy(), k2["tier"].to_numpy()
    X_emb = E
    X_both = np.hstack([E, k2[feat_cols].to_numpy(float)])
    mk_lr = lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                  LogisticRegression(C=0.02, class_weight="balanced", max_iter=400))
    emb_folds = []
    for task, y in (("7class", k2["behavior"].to_numpy()), ("coarse", k2["coarse"].to_numpy())):
        for nm, X in (("frozen_emb_LR", X_emb), ("frozen_emb+geometry_LR", X_both)):
            f, P = lodo(X, y, g2, t2, mk_lr, f"{task}:{nm}")
            emb_folds.append(f); preds[f"{task}:{nm}"] = P
    display(summarize(all_folds + emb_folds))
    cm, rec = recall_table(k2["behavior"].to_numpy(), preds["7class:frozen_emb+geometry_LR"], t2, CLASSES7)
    display(cm); display(rec.round(2))

## 5. What to do with the numbers

Compare these rows against the v1 fine-tune (validation macro-F1 0.286 on day 2019-11-11, accuracy 0.323, with only 1,684 validation rows from one day):

- If geometry or frozen-embedding models beat `majority` and `stratified_random` clearly, the signal is there and a fine-tuned model has a fair target to beat.
- If `sleep` vs `lying` stays confused (it was 147 of 223 lying rows predicted as sleep in v1), merge them as `resting` for the headline result and say so. From above they look nearly identical.
- If `eat`, `drink` and `standing` stay weak without pen position, add zone features from the camera layout (feeder, drinker) and report them as camera-specific rules.
- Only then fine-tune: freeze the backbone for 1 to 2 epochs, use label smoothing and stronger augmentation, subsample every 3rd to 4th step per pig (neighbours 0.5 s apart are near duplicates), and score it with the same leave-one-day-out loop.

Still open for the project: a fine-tuned pig detector with ByteTrack (so crops come from detections, not ground truth), the fps ablation, INT8 vs FP32, and byte logging.